In [7]:
from constants import ORACLE_SKILLS, SKIP_DIRS
from pathlib import Path

docs_root = Path(ORACLE_SKILLS)
markdown_files = []
for path in sorted(docs_root.rglob("*.md")):
  if any(part in SKIP_DIRS for part in path.parts):
    continue
  if path.is_file():
    markdown_files.append(path)

for filepath in markdown_files[:10]:
  print(str(filepath))

skills-main\db\admin\backup-recovery.md
skills-main\db\admin\dataguard.md
skills-main\db\admin\redo-log-management.md
skills-main\db\admin\rman-basics.md
skills-main\db\admin\undo-management.md
skills-main\db\admin\user-management.md
skills-main\db\agent\client-identification.md
skills-main\db\agent\destructive-op-guards.md
skills-main\db\agent\idempotency-patterns.md
skills-main\db\agent\intent-disambiguation.md


In [2]:
# BM25 on the whole markdown files
# Clearly not a good idea as you will not inject a full markdown file in the prompt of you data model call
# The obvious strategy is to inject only (best) chunks related to the question
import bm25s
import Stemmer

corpus = []
for path in markdown_files:
  try:
    content = path.read_text(encoding="utf-8", errors="ignore").strip().lower()
    corpus.append(content)
  except Exception as e:
    print(f"Error reading {path}: {e}")

# Stemmer
stemmer = Stemmer.Stemmer("english")
# Tokenize the corpus and index it
metadata_corpus = []
for i in range(len(corpus)):
  # print(f"Document {markdown_files[i]} score: {doc_scores[i]}")
  metadata_corpus.append({ "file": str(markdown_files[i]), "text": corpus[i] })
# print (metadata_corpus[0])
corpus_tokens = bm25s.tokenize([doc["text"] for doc in metadata_corpus], stopwords="en", stemmer=stemmer)
retriever = bm25s.BM25(corpus=metadata_corpus)
retriever.index(corpus_tokens)

# You can now search the corpus with a query
query = "How to generate an AWR report in HTML format ?"
query_tokens = bm25s.tokenize(query,  stemmer=stemmer)
results, scores = retriever.retrieve(query_tokens, show_progress=False, k=20)

# print(results)
# print(scores)
# results_2 = []
for i in range(results.shape[1]):
  doc, score = results[0, i], scores[0, i]
  # results_2.append({ "file": str(markdown_files[i]), "score": float(score) })
  print(f"Rank {i+1} (score: {score:.2f}), File: {doc["file"]}")

# sorted_results = sorted(results_2, key=lambda x: x["score"], reverse=True)
# print(json.dumps(sorted_results, indent=2, ensure_ascii=False))

resource module not available on Windows


Split strings:   0%|          | 0/156 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/156 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/156 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/156 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

Rank 1 (score: 4.72), File: skills-main\db\sqlcl\sqlcl-awr.md
Rank 2 (score: 4.30), File: skills-main\db\monitoring\top-sql-queries.md
Rank 3 (score: 3.75), File: skills-main\db\sqlcl\sqlcl-mcp-server.md
Rank 4 (score: 3.59), File: skills-main\db\SKILL.md
Rank 5 (score: 3.37), File: skills-main\db\performance\explain-plan.md
Rank 6 (score: 3.30), File: skills-main\db\performance\ash-analysis.md
Rank 7 (score: 3.27), File: skills-main\db\monitoring\space-management.md
Rank 8 (score: 3.20), File: skills-main\db\agent\client-identification.md
Rank 9 (score: 3.18), File: skills-main\db\performance\awr-reports.md
Rank 10 (score: 3.06), File: skills-main\db\sqlcl\sqlcl-background-jobs.md
Rank 11 (score: 2.88), File: skills-main\db\agent\nl-to-sql-patterns.md
Rank 12 (score: 2.84), File: skills-main\db\monitoring\health-monitor.md
Rank 13 (score: 2.76), File: skills-main\db\sqlcl\sqlcl-formatting.md
Rank 14 (score: 2.75), File: skills-main\db\architecture\exadata-features.md
Rank 15 (score: 2

In [9]:
# BM25 on te chunks of markdown files
import bm25s
import Stemmer
from BM25_and_embeddings import chunks_from_markdown_files

# Stemmer
stemmer = Stemmer.Stemmer("english")

# chunks
chunks = chunks_from_markdown_files()

corpus_tokens = bm25s.tokenize(texts = [item["chunk"] for item in chunks], show_progress = False, stopwords = "en", stemmer = stemmer)
retriever = bm25s.BM25(corpus = chunks)
retriever.index(corpus = corpus_tokens)

# You can now search the corpus with a query
query = "How to generate an AWR report in HTML format ?"
query_tokens = bm25s.tokenize(texts = [query], show_progress = False, stemmer = stemmer)
results, scores = retriever.retrieve(query_tokens = query_tokens, corpus = chunks, show_progress = False, k = 20)

print(results)
print(scores)
# results_2 = []
for i in range(results.shape[1]):
  result, score = results[0, i], scores[0, i]
  # results_2.append({ "file": str(markdown_files[i]), "score": float(score) })
  print(f"--------------------------------------------------------------------------------\nRank {i+1} (score: {score:.2f}), File: {result['source']}\nChunk: {result['chunk']}")

# sorted_results = sorted(results_2, key=lambda x: x["score"], reverse=True)
# print(json.dumps(sorted_results, indent=2, ensure_ascii=False))

BM25S Count Tokens:   0%|          | 0/3578 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/3578 [00:00<?, ?it/s]

[[{'id': 3273, 'source': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'section_id': 5, 'chunk_id': 0, 'header_path': ['SQLcl AWR Command', 'Common Workflows', 'Diagnose a recent performance problem'], 'chunk': '```sql\n-- List snapshots to find the relevant time window\nawr list snapshots\n\n-- Generate an HTML report for that window (e.g., snapshots 142 to 145)\nawr create html 142 145\n```  \nOpen the resulting `.html` file in a browser for the full formatted report including Top SQL, wait events, load profile, and instance efficiency metrics.'}
  {'id': 3279, 'source': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'section_id': 11, 'chunk_id': 0, 'header_path': ['SQLcl AWR Command', 'Best Practices'], 'chunk': '- Run `awr list snapshots` before generating a report to confirm the snapshot IDs exist and span the time window you want.\n- Use HTML format for human review; use text format when the output needs to be parsed programmatically or sent in plain-text email.\n- Keep snapshot intervals sho

In [ ]:
import importlib
import BM25_and_embeddings
# Sometimes te Notebook has some problem taking the last saved version of the library, so we force the reload of the library to be sure to have the last version of the code
importlib.reload(BM25_and_embeddings)
from BM25_and_embeddings import bm25_retriever_from_chunks, chunks_from_markdown_files, bm25_score

chunks = chunks_from_markdown_files()
query = "How to generate an AWR report in HTML format ?"
bm25_retriever = bm25_retriever_from_chunks(chunks)
bm25_score_dict = bm25_score(query, chunks, bm25_retriever, 100)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/3578 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/3578 [00:00<?, ?it/s]

In [11]:
retrieved_context = ""
for item in sorted(bm25_score_dict, key=lambda x: x.get("bm25_score", 0.0), reverse=True)[:10]:
  retrieved_context += f"[{item.get('id')}]\nsource: {item.get('source')}\nsection: {" > ".join(item.get('header_path', []))}\ncontent: {item.get('chunk')}\n\n"

print(retrieved_context)

[3273]
source: skills-main\db\sqlcl\sqlcl-awr.md
section: SQLcl AWR Command > Common Workflows > Diagnose a recent performance problem
content: ```sql
-- List snapshots to find the relevant time window
awr list snapshots

-- Generate an HTML report for that window (e.g., snapshots 142 to 145)
awr create html 142 145
```  
Open the resulting `.html` file in a browser for the full formatted report including Top SQL, wait events, load profile, and instance efficiency metrics.

[3279]
source: skills-main\db\sqlcl\sqlcl-awr.md
section: SQLcl AWR Command > Best Practices
content: - Run `awr list snapshots` before generating a report to confirm the snapshot IDs exist and span the time window you want.
- Use HTML format for human review; use text format when the output needs to be parsed programmatically or sent in plain-text email.
- Keep snapshot intervals short (15–30 minutes) during active problem investigation so you can isolate the exact period when performance degraded.
- The default AW